## Prueba de conexión a la DB

In [1]:
from src.db.connector import DatabaseConnector

db = DatabaseConnector()
if db.test_connection():
    print("✅ Conexión a la base de datos establecida correctamente.")
else:
    print("❌ No se pudo conectar a la base de datos.")

✅ Conexión a la base de datos establecida correctamente.


## Prueba de query

In [2]:
# Ejecutamos una consulta simple para verificar que se pueden traer datos desde la tabla 'products'
from src.db.connector import DatabaseConnector

db = DatabaseConnector()
df = db.execute_query("SELECT * FROM products LIMIT 5")
df

,product_id,name,price,category_id,modify_date,product_class,resistant,is_allergic,vitality_days
0,1,Flour - Whole Wheat,74.30,3,None,Medium,Durable,Unknown,0
1,2,Cookie Chocolate Chip With,91.23,3,None,Medium,Unknown,Unknown,0
2,3,Onions - Cippolini,9.14,9,None,Medium,Weak,FALSE,111
3,4,Sauce - Gravy; Au Jus; Mix,54.31,9,None,Medium,Durable,Unknown,0
4,5,Artichokes - Jerusalem,65.48,2,None,Low,Durable,TRUE,27


## Uso de patrones de diseño

In [3]:
# Demostramos que el patrón Singleton está funcionando: ambas instancias son la misma
db1 = DatabaseConnector()
db2 = DatabaseConnector()

print("¿Es la misma instancia?", db1 is db2)  # Debería devolver True

¿Es la misma instancia? True


## Ejecutamos los test unitarios

In [4]:
import pytest

resultado = pytest.main(["-s", "tests/"])
if resultado == 0:
    print("✅ Todas las pruebas pasaron correctamente.")
else:
    print("❌ Algunas pruebas fallaron. Revisa los logs para más detalles.")

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.3.5, pluggy-1.6.0
rootdir: /Users/nacho/Documents/Repositorios/accenture-pi
collected 7 items

tests/test_category.py ✅ test_category_creation pasó correctamente.
.
tests/test_connection.py ✅ test_connection pasó correctamente. Conexión establecida.
.
tests/test_customer.py ✅ test_customer_full_name pasó correctamente.
.
tests/test_execute_query.py ✅ test_execute_query pasó correctamente.
.
tests/test_load_categories.py ✅ test_load_categories_from_csv pasó correctamente.
.
tests/test_product.py ✅ Precio con descuento aplicado: 90.0
.
tests/test_sale.py ✅ Relaciones en la venta correctamente establecidas
.

============================== 7 passed in 0.04s ===============================
✅ Todas las pruebas pasaron correctamente.


# Queries avanzadas

## Compras por cliente (uso de CTE)

Esta consulta muestra el total de productos comprados por cada cliente. Se utiliza una **CTE (Common Table Expression)** llamada `compras_por_cliente` para calcular la suma total de unidades (`quantity`) agrupadas por `customer_id` a partir de la tabla `sales`.

El uso de la CTE nos permite separar esta lógica de agregación del filtrado posterior, mejorando la legibilidad del código SQL. Luego, seleccionamos solo aquellos clientes que realizaron más de 100 compras.

Este tipo de análisis permite identificar a los clientes más activos, útil para estrategias comerciales o de fidelización. Ejecutamos esta consulta desde Python mediante una función que utiliza el conector a la base de datos.

In [5]:
from src.queries.report_queries import compras_por_cliente

df_compras = compras_por_cliente(100)
display(df_compras)

,customer_id,total_compras
0,81679,105
1,94115,120
2,94696,120
3,95242,125
4,98410,125


## Ranking de productos por categoría (uso de función ventana RANK)

En esta consulta queremos conocer cuáles son los productos más vendidos dentro de cada categoría. Para ello:

- Usamos una **CTE** llamada `ventas_agrupadas` para sumar la cantidad total de unidades vendidas (`quantity`) por producto (`product_id`) y categoría (`category_id`).
- Luego aplicamos una **función ventana `RANK()`** dentro de una segunda CTE (`ranking_cte`) para asignar un orden dentro de cada categoría según el total de ventas.
- Finalmente, filtramos los tres productos mejor posicionados por categoría (top 3), lo que permite identificar fácilmente los productos más exitosos en cada segmento.

Este enfoque es muy útil para reportes de negocio o dashboards internos. Ejecutamos esta lógica desde Python a través de una función dedicada.

In [7]:
from src.queries.report_queries import ranking_productos_por_categoria

df_ranking = ranking_productos_por_categoria()
display(df_ranking)

,category_id,product_id,product_name,total_quantity,rnk
0,1,156,Sprouts - Alfalfa,1910,1
1,1,250,Soup - Campbells; Beef Barley,1858,2
2,1,451,Soup - Campbells Tomato Ravioli,1728,3
3,2,383,Cake - Box Window 10x10x2.5,1882,1
4,2,212,Curry Paste - Madras,1811,2
5,2,418,Truffle Cups - Brown,1671,3
6,3,65,Brandy - Bar,1904,1
7,3,177,Coconut - Shredded; Sweet,1896,2
8,3,424,Vinegar - Tarragon,1830,3
9,4,252,Nantucket - Pomegranate Pear,1657,1
